# RAG Document Assistant — Day 1: Foundations
### Loading documents, chunking text, and understanding embeddings — the raw materials of every RAG system.

**Stack:** Google Gemini 2.5 Flash + Gemini Embeddings + LangChain + ChromaDB

**Today's goal:** No vector database yet, no Q&A yet. Today is purely about understanding and testing the *first half* of the RAG pipeline — Load → Split → Embed — one piece at a time, so nothing feels like magic later.

---

## Step 1 — Install Dependencies

In [1]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters chromadb pypdf python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Setup API Key

We reuse the same `.env` pattern from the email agent project. Only one key is needed today: `GEMINI_API_KEY`.

If you don't have a `.env` file in this folder yet, create one with:
```
GEMINI_API_KEY=your_key_here
```

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
print("API key loaded successfully.")

API key loaded successfully.


## Step 3 — Import Everything

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18752\2328665654.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader


## Step 4 — Load Your Documents

RAG needs *something* to retrieve from. Create a folder called `data/` next to this notebook and drop in a few PDFs or `.txt` files — your notes, a resume, an assignment, a research paper, anything.

`DirectoryLoader` scans a folder and loads every matching file into LangChain `Document` objects — each one holding raw text + metadata (like which file/page it came from).

> If you don't have files ready yet, download any free PDF (e.g. a short research paper or an ebook chapter) and drop it into `data/` before running this cell.

In [4]:
DATA_PATH = "data"

loader = DirectoryLoader(
    DATA_PATH,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} document page(s) from '{DATA_PATH}'")
if documents:
    print("\n--- Preview of first page ---")
    print(documents[0].page_content[:500])
    print("\n--- Metadata ---")
    print(documents[0].metadata)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 82 0 (offset 0)
Ignoring wrong pointing object 137 0 (offset 0)
Ignoring wrong pointing object 139 0 (offset 0)
Ignoring wrong pointing object 190 0 (offset 0)
Ignoring wrong pointing object 212 0 (offset 0)
Ignoring wrong pointing object 214 0 (offset 0)
Ignoring wrong pointing object 231 0 (offset 0)
Ignoring wrong pointing object 249 0 (offset 0)
Ignoring wrong pointing object 300 0 (offset 0)
Ignoring wrong pointing object 311 0 (offset 0)
Ignoring wrong pointing object 313 0 (offset 0)
Ignoring wrong pointing object 361 0 (offset 0)
Ignoring wrong pointing object 372 0 (offset 0)
Ignoring wrong pointing object 374 0 (offset 0)
Ignoring wrong pointing object 386 0 (offset 0)
Ignoring wrong pointing object 440 0 (offset 0)
Ignoring wrong pointing object 453 0 (offset 0)
Ignoring wrong pointing object 491 0 (offset 0

Loaded 221 document page(s) from 'data'

--- Preview of first page ---
OPERATING SYSTEMS  
UE24CS242B
Introduction, Computer System 
Organization
Pavan A C
Department of Computer Science & Engineering

--- Metadata ---
{'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext, AppendMode 1.1', 'creator': 'PyPDF', 'creationdate': "D:20260120060059Z00'00'", 'moddate': "D:20260122051606Z00'00'", 'source': 'data\\OS unit 1 slides 2026 full.pdf', 'total_pages': 221, 'page': 0, 'page_label': '1'}


## Step 5 — Split Text into Chunks (Why Chunking Matters)

You can't just hand the model an entire 40-page PDF and say "here, use this" — LLMs have a limited context window, and stuffing huge raw pages in hurts retrieval precision anyway.

Instead, we **split documents into small overlapping chunks**. Each chunk becomes one retrievable unit later.

- `chunk_size=1000` → roughly 1000 characters per chunk
- `chunk_overlap=200` → each chunk shares 200 characters with the next one, so we don't accidentally cut a sentence or idea in half at a chunk boundary

`RecursiveCharacterTextSplitter` is "recursive" because it tries to split on paragraph breaks first, then sentences, then words — only falling back to a hard character cut if it has to. This keeps chunks semantically coherent instead of arbitrarily sliced.

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document page(s) into {len(chunks)} chunk(s)")
if chunks:
    print("\n--- Preview of chunk 0 ---")
    print(chunks[0].page_content)
    print(f"\nChunk length: {len(chunks[0].page_content)} characters")

Split 221 document page(s) into 221 chunk(s)

--- Preview of chunk 0 ---
OPERATING SYSTEMS  
UE24CS242B
Introduction, Computer System 
Organization
Pavan A C
Department of Computer Science & Engineering

Chunk length: 129 characters


## Step 6 — Generate Embeddings (Turning Text into Vectors)

An embedding model converts text into a list of numbers (a **vector**) that represents its *meaning*. Texts with similar meaning end up with vectors that are close together in that vector space — that's the entire trick behind semantic search.

We're **not** building the vector database yet (that's Day 2). Today, just embed a single chunk so you can *see* what an embedding actually looks like — most people use RAG for months without ever looking at a raw vector.

In [7]:
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

sample_text = chunks[0].page_content if chunks else "LangChain makes it easy to build LLM applications."

vector = embeddings_model.embed_query(sample_text)

print(f"Embedding vector length: {len(vector)} dimensions")
print(f"First 10 values: {vector[:10]}")

Embedding vector length: 3072 dimensions
First 10 values: [0.022703245, -0.02803864, 0.025048973, -0.043494564, -0.038556136, 0.017199421, 0.015742067, 0.0034135615, -0.0020474007, 0.0064925794]


## Step 7 — Quick Sanity Check: Similar Text → Similar Vectors

Let's prove the core idea of embeddings to ourselves before moving on: two sentences with similar *meaning* should produce vectors that are close together, even if they don't share many exact words.

In [8]:
import numpy as np

def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sentence_a = "The cat sat on the warm windowsill in the sun."
sentence_b = "A kitten was relaxing on a sunny window ledge."
sentence_c = "Quarterly revenue grew by twelve percent this year."

vec_a = embeddings_model.embed_query(sentence_a)
vec_b = embeddings_model.embed_query(sentence_b)
vec_c = embeddings_model.embed_query(sentence_c)

print("Similarity (A, B) — meaning-related sentences:", round(cosine_similarity(vec_a, vec_b), 4))
print("Similarity (A, C) — unrelated sentences:      ", round(cosine_similarity(vec_a, vec_c), 4))

Similarity (A, B) — meaning-related sentences: 0.8397
Similarity (A, C) — unrelated sentences:       0.5626


---
## Day 1 Wrap-Up

Today you built and tested, piece by piece, the **ingestion half** of a RAG pipeline:

`Raw files → Document objects → Chunks → Embeddings → (proof that embeddings capture meaning)`

Nothing is stored permanently yet — every time you rerun this notebook, you'd re-embed everything from scratch. That's exactly the gap Day 2 closes.

**Tomorrow (Day 2):** we store these chunk embeddings in a real vector database (Chroma), persist it to disk, and write a retrieval function that takes a question and returns the most relevant chunks — the actual "R" in RAG.

See `README_Day1.md` for the full write-up of today's concepts.